In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import scipy
import sklearn
import math

## Utilities

We define:
1. Neural networks.
2. `fit` function to train the networks.
3. Helper visualization functions.

In [ ]:
from typing import Callable
from torch import nn
from torch.optim import Optimizer
from torch.utils.data import Dataset
from typing import Literal
from tqdm import tqdm


class MLP(nn.Module):
    def __init__(self, dim_in: int, dim_hidden: int, dim_out: int):
        super().__init__()
        self.dim_in = dim_in
        self.dim_hidden = dim_hidden
        self.dim_out = dim_out
        self.net = nn.Sequential(
            nn.Linear(self.dim_in, self.dim_hidden),
            nn.SiLU(),
            nn.Linear(self.dim_hidden, self.dim_hidden),
            nn.SiLU(),
            nn.Linear(self.dim_hidden, self.dim_out),
        )

    def forward(self, x: torch.Tensor):
        return self.net(x)


def fit(
    model: nn.Module,
    *,
    loss_fn: Callable[[torch.Tensor, torch.Tensor], torch.Tensor],
    optimizer: Optimizer,
    train_data: Dataset,
    valid_data: Dataset,
    max_n_epochs: int = 100,
    patience: int = 5,
    log_freq: float = 0.1,
):
    train_dataloader = torch.utils.data.DataLoader(
        train_data, batch_size=32, shuffle=True, drop_last=False
    )
    valid_dataloader = torch.utils.data.DataLoader(
        valid_data, batch_size=32, shuffle=False, drop_last=False
    )

    def step(mode: Literal["train", "valid"]):
        dataloader = train_dataloader if mode == "train" else valid_dataloader
        losses: list[float] = []
        with torch.set_grad_enabled(mode == "train"):
            model.train(mode == "train")
            for x, y in dataloader:
                pred = model(x)
                loss = loss_fn(pred, y)
                losses.append(loss.item())

                if mode == "train":
                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()
        return torch.tensor(losses).mean()

    best_valid = float("inf")
    best_ckpt = model.state_dict()
    cur_patience = patience
    for epoch in tqdm(range(max_n_epochs)):
        train_loss = step("train")
        valid_loss = step("valid")
        if epoch % int(log_freq * max_n_epochs) == 0 or epoch + 1 == max_n_epochs:
            print(f"Epoch: {epoch}, train loss: {train_loss}, valid loss: {valid_loss}")
        if valid_loss < best_valid:
            best_valid = valid_loss
            cur_patience = patience
            best_ckpt = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            cur_patience -= 1
        if cur_patience < 0:
            break

    model.load_state_dict(best_ckpt)
    print("Best ckpt valid loss: ", step("valid"))

In [ ]:
from typing import Callable


def viz_img(
    value_fn: Callable[[torch.Tensor], torch.Tensor], ax, cax=None, cax_label=None
):
    global G_x1_min_display, G_x1_max_display, G_x2_min_display, G_x2_max_display
    GRID_SIZE = 500
    X1, X2 = torch.meshgrid(
        torch.linspace(G_x1_min_display, G_x1_max_display, GRID_SIZE),
        torch.linspace(G_x2_min_display, G_x2_max_display, GRID_SIZE),
        indexing="xy",
    )

    points = torch.stack([X1.ravel(), X2.ravel()], axis=-1)
    v = value_fn(points).reshape(GRID_SIZE, GRID_SIZE)
    im_probability = ax.imshow(
        v,
        extent=[G_x1_min_display, G_x1_max_display, G_x2_min_display, G_x2_max_display],
        origin="lower",
        interpolation="bilinear",
        cmap="Blues",
    )
    plt.colorbar(im_probability, cax=cax, label=cax_label)


def viz_scatter(
    samples: torch.Tensor,
    value_fn: Callable[[torch.Tensor], torch.Tensor],
    ax,
    cax=None,
    cax_label=None,
):
    v = value_fn(samples)
    im = ax.scatter(
        samples[:, 0],
        samples[:, 1],
        s=5,
        c=v,
        alpha=1.0,
    )
    plt.colorbar(im, cax=cax, label=cax_label)

# Problem Setup
In this notebook, we operate in a synthetic 2D continuous space. Instead of working with complex image pixels, this low-dimensional setup allows us to easily visualize how diffusion models manipulate probability densities, vector fields, and sampling trajectories.

----

We define a custom target distribution $p_X(x)$ called ThickSineDensity.

The Geometry: The mass of the distribution is concentrated along a sinusoidal curve $x_2 = \sin(\omega \pi x_1)$ bounded within a specific horizontal window $[x_1^{\text{min}}, x_1^{\text{max}}]$.

The Noise/Thickness: The distribution has a uniform thickness defined by $\sigma$. Points are distributed vertically around the sine wave according to a cosine-shaped probability density function, dropping cleanly to zero outside the boundary ($r > 1.0$).

----

Our goal is to treat this distribution as the unknown "true data distribution". The generative model will only have access to a finite set of empirical samples generated from this distribution, mimicking real-world scenarios.

## Target Distribution

### Distribution definition

In [ ]:
from torch.distributions import Distribution, constraints


class ThickSineDensity(Distribution):
    arg_constraints = {
        "omega": constraints.positive,
        "sigma": constraints.positive,
        "x1_min": constraints.real,
        "x1_max": constraints.real,
    }
    has_rsample = False

    def __init__(
        self,
        omega: float = 2.0,
        sigma: float = 0.08,
        x1_min: float = -2.0,
        x1_max: float = 2.0,
    ):
        self.omega = torch.as_tensor(omega, dtype=torch.float32)
        self.sigma = torch.as_tensor(sigma, dtype=torch.float32)
        self.x1_min = torch.as_tensor(x1_min, dtype=torch.float32)
        self.x1_max = torch.as_tensor(x1_max, dtype=torch.float32)
        super().__init__(
            batch_shape=torch.Size([]),
            event_shape=torch.Size([2]),
            validate_args=True,
        )

    def _mean_curve(self, t: torch.Tensor) -> torch.Tensor:
        return torch.sin(self.omega * math.pi * t)

    def log_prob(self, x: torch.Tensor) -> torch.Tensor:
        x1 = x[..., 0]
        x2 = x[..., 1]

        mu = self._mean_curve(x1)
        r = (x2 - mu).abs() / self.sigma

        inside = (x1 >= self.x1_min) & (x1 <= self.x1_max) & (r <= 1.0)

        w = torch.cos((math.pi / 2) * r.clamp(max=1.0))
        log_normalizer = torch.log(
            (self.x1_max - self.x1_min) * (4.0 * self.sigma / math.pi)
        )

        return torch.where(
            inside,
            torch.log(w) - log_normalizer,
            torch.full_like(w, float("-inf")),
        )

    def sample(self, sample_shape: torch.Size = torch.Size()) -> torch.Tensor:
        shape = self._extended_shape(sample_shape)
        n = shape[:-1].numel()

        with torch.no_grad():
            x1 = torch.empty(n).uniform_(self.x1_min.item(), self.x1_max.item())
            x2 = torch.empty(n)
            mu = self._mean_curve(x1)

            pending = torch.ones(n, dtype=torch.bool)
            while pending.any():
                n_pending = int(pending.sum())

                delta = torch.empty(n_pending).uniform_(
                    -self.sigma.item(), self.sigma.item()
                )
                accept_prob = torch.cos((math.pi / 2) * delta.abs() / self.sigma)
                accepted = torch.rand(n_pending) < accept_prob

                if accepted.any():
                    idx = pending.nonzero(as_tuple=False).squeeze(1)[accepted]
                    x2[idx] = mu[idx] + delta[accepted]
                    pending[idx] = False

        return torch.stack([x1, x2], dim=-1).reshape(shape)

    def viz_support(self, ax):
        t = torch.linspace(self.x1_min, self.x1_max, 4000)
        curve = self._mean_curve(t)
        curve_lower = curve - self.sigma
        curve_upper = curve + self.sigma
        ax.plot(t, curve_upper, color="navy", linestyle="--", linewidth=2)
        ax.plot(t, curve_lower, color="navy", linestyle="--", linewidth=2)

### Global variables definition

In [ ]:
omega = 0.5
sigma = 0.5
x1_min = -4.0
x1_max = 2.0
x2_min = -1.0 - sigma
x2_max = 1.0 + sigma
N_gen = 400
N_clf = 50

G_x1_min_display = x1_min - 1
G_x1_max_display = x1_max + 1
G_x2_min_display = x2_min - 1
G_x2_max_display = x2_max + 1

G_gt_distribution = ThickSineDensity(
    omega=omega, sigma=sigma, x1_min=x1_min, x1_max=x1_max
)

### Visualization

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(15, 5), gridspec_kw={"width_ratios": [1.0, 1.0]})

ax_probability = axs[0]
ax_samples = axs[1]

ax_samples.set_title("Samples")
ax_probability.set_title("Probability")

viz_img(
    lambda x: G_gt_distribution.log_prob(x).exp(), ax_probability, cax_label="density"
)

samples = G_gt_distribution.sample((N_gen,))
ax_samples.scatter(samples[:, 0], samples[:, 1], alpha=0.2)

for ax in [ax_samples, ax_probability]:
    G_gt_distribution.viz_support(ax)
    ax.set_xlim(G_x1_min_display, G_x1_max_display)
    ax.set_ylim(G_x2_min_display, G_x2_max_display)
    ax.set_xlabel(r"$x_1$")
    ax.set_ylabel(r"$x_2$")
    ax.set_aspect("equal")


plt.tight_layout()
plt.show()

## Objective-Driven Generation via Reward Functions

In standard generative modeling, our only goal is to mimic the training data distribution $p_X(x)$. However, in many real-world tasks (e.g., drug discovery, material design, or reinforcement learning alignment), we don't just want any valid sample. We want high-utility samples.

The Challenge: We want to steer our generator to sample from regions where both the ground-truth data density $p_X(x)$ is high (so the sample is "realistic") and the reward $R(x)$ is maximized.

### Example reward definition

In [ ]:
G_reward_fn = lambda x: (x[..., -1] ** 2) + x[..., 0].abs()

### Visualization

In [ ]:
samples = G_gt_distribution.sample((1000,))
viz_scatter(samples, G_reward_fn, plt.gca(), cax_label="rewards")
G_gt_distribution.viz_support(plt.gca())

## Metrics

To measure the performance of the generator and guidance we introduce several empirical measures.

### Definitions

__Reward optimization metrics__

- **`rewards_mean`** — Average reward across all generated samples, with invalid samples contributing zero.
  - *Strengths*: Single summary number for overall optimization quality; easy to track across runs.
  - *Limitations*: In optimized generation we usually care about the model's ability to produce very high-reward samples, not just generally okay ones.

- **`rewards_max`** — The single best reward value found among the generated samples.
  - *Strengths*: Directly answers "what's the best thing this model can produce"; useful for best-case/exploration-oriented evaluation.
  - *Limitations*: High variance — a single lucky (or unlucky) sample dominates; not representative of typical behavior; can be gamed by generating more samples.

- **`rewards_q95`** — Reward value at the 95th percentile, capturing the quality of the best-performing tail of samples.
  - *Strengths*: Useful when the goal is to find a few very good samples (e.g., drug discovery, design search) rather than optimize the average case; more robust than the mean to the bulk of low/zero-reward samples.
  - *Limitations*: Ignores most of the distribution.

---
__Feasibility metrics__

- **`MMD^2`** — Squared Maximum Mean Discrepancy between generated samples and a reference dataset, capturing distributional similarity.
  - *Strengths*: Principled kernel-based two-sample statistic capturing differences in location and shape, not just first/second moments; lower values mean closer match to the reference.
  - *Limitations*: Sensitive to kernel bandwidth (`gamma`, fixed at 1.0 here, may not suit all data scales); quadratic cost in sample size; hard to interpret in absolute terms (relative/comparative, not an intuitive distance).

- **`validity_mean`** — Fraction of generated samples that fall within the support of the reference distribution (i.e., have nonzero probability).
  - *Strengths*: Simple, interpretable as a percentage; measures feasibility/constraint satisfaction independent of reward magnitude.
  - *Limitations*: Only practical when an oracle is available to evaluate whether a sample is valid.
---
__Diversity metrics__

- **`pairwisedistance_mean`** — Average pairwise Euclidean distance between generated samples, indicating how spread out they are.
  - *Strengths*: Simple, intuitive measure of intra-batch diversity/mode coverage; helps detect mode collapse (collapsed samples cluster together, giving a low mean distance).
  - *Limitations*: Ignores the reference distribution, so high diversity isn't necessarily "good" if it covers implausible/low-density regions; quadratic cost; sensitive to input scale, so not directly comparable across domains.

In [ ]:
from scipy.spatial.distance import pdist
from sklearn.metrics.pairwise import rbf_kernel
from dataclasses import dataclass, field
from typing import Callable


@dataclass
class SampledDistributionToOptimize:
    unlabeled_x: torch.Tensor
    labeled_x: torch.Tensor
    labeled_y: torch.Tensor
    compute_metrics: Callable[[torch.Tensor], dict[str, float]] = field(repr=False)


class DistributionToOptimize:
    def __init__(
        self,
        distribution: Distribution,
        reward_fn: Callable[[torch.Tensor], torch.Tensor],
    ):
        self.distribution = distribution
        self.reward_fn = reward_fn

    def sample_train_data(
        self, N_gen: int, N_clf: int
    ) -> SampledDistributionToOptimize:
        train_x_gen = self.distribution.sample((N_gen,))

        def compute_metrics(samples: torch.Tensor):
            assert samples.shape[-1] == 2
            samples = samples.reshape(-1, samples.shape[-1])

            probs = self.distribution.log_prob(samples).exp()
            rewards = self.reward_fn(samples)
            valid_rewards = self.reward_fn(samples) * (probs > 0)
            pairwise_dists = pdist(samples)
            mmd_sq = (
                rbf_kernel(samples, samples, gamma=1.0).mean()
                + rbf_kernel(train_x_gen, train_x_gen, gamma=1.0).mean()
                - 2 * rbf_kernel(samples, train_x_gen, gamma=1.0).mean()
            )

            return {
                "rewards_mean": rewards.mean().item(),
                "rewards_max": rewards.max().item(),
                "rewards_q95": torch.quantile(rewards, 0.95).item(),
                "valid_rewards_mean": valid_rewards.mean().item(),
                "valid_rewards_max": valid_rewards.max().item(),
                "valid_rewards_q95": torch.quantile(valid_rewards, 0.95).item(),
                "MMD^2": mmd_sq.item(),
                "validity_mean": (probs > 0).float().mean().item(),
                "pairwisedistance_mean": pairwise_dists.mean().item(),
            }

        return SampledDistributionToOptimize(
            unlabeled_x=train_x_gen,
            labeled_x=train_x_gen[:N_clf],
            labeled_y=self.reward_fn(train_x_gen[:N_clf]),
            compute_metrics=compute_metrics,
        )

### Global variables definition

In [ ]:
G_distribution_to_optimize = DistributionToOptimize(G_gt_distribution, G_reward_fn)
G_sampled_distribution_to_optimize = G_distribution_to_optimize.sample_train_data(
    N_gen, N_clf
)

### Example metrics computation

In [ ]:
G_sampled_distribution_to_optimize.compute_metrics(
    G_sampled_distribution_to_optimize.unlabeled_x
)

# Continuous Diffusion

### Score Matching

Select any drift schedule $a_t$ and a noise schedule $b_t$. Assume $x_0\sim p_X$ and define the noised versions of clean data in the following way:

$$x_t = a_t x_0 + b_t \varepsilon_t,$$

where the noise marginals follow $\varepsilon_t\sim \mathcal{N}(0,I)$. This way, the noised data marginals follow $x_t\sim p_t = \mathcal{N}(a_t x_0; b_t^2 I)$.

One can learn to predict the true data directly:
$$\mathcal{L}_{x}(\theta) = \mathbb{E}_{t, x_0, \varepsilon_t} \left[ \| x_{\theta}(x_t, t) - x_0 \|^2 \right],$$

or equivalently, the noise vactor:

$$\mathcal{L}_{\epsilon}(\theta) = \mathbb{E}_{t, x_0, \varepsilon_t} \left[ \| \epsilon_{\theta}(x_t, t) - \varepsilon_t \|^2 \right].$$

---

### Equivalence of $x_\theta$ and $\epsilon_\theta$

Note that the minimizer of $\mathcal{L}_x$ is $x_{\theta^*}(x_t, t) = \mathbb{E}[x_0 | x_t]$, and the minimizer of $\mathcal{L}_\epsilon$ is $\epsilon_{\theta^*}(x_t, t) = \mathbb{E}[\varepsilon_t | x_t]$. Because $x_t = a_t x_0 + b_t \varepsilon_t$, taking the conditional expectation wrt. $x_t$ we obtain
$$x_t = a_t \mathbb{E}[x_0|x_t] + b_t \mathbb{E}[\varepsilon_t|x_t],$$
therefore, given $x_t$ one can easily translate between $\mathbb{E}[x_0 | x_t]$ and $\mathbb{E}[\varepsilon_t | x_t]$.

---

### Tweedie's Formula & The Score Function

$$\nabla_{x_t} \log p_\sigma(x_t) = -\frac{1}{b_t^2}\left(x-a_t\mathbb{E}[x_0 | x]\right)$$

---

Proof: 

Note that
$$\nabla_{x} \log p_t(x) = \frac{\nabla_x p_t(x)}{p_t(x)} = \frac{\int_{x_0 } \nabla_x p_t(x|x_0) p_t(x_0)dx_0}{p_t(x)}$$
and
$$\nabla_x p_t(x|x_0) = -\frac{x-a_t x_0}{b_t^2} p_t(x|x_0).$$

Therefore 

$$\nabla_{x} \log p_t(x) = \frac{\int_{x_0 } -\frac{x-a_t x_0}{b_t^2} p_t(x|x_0) p_t(x_0)dx_0}{p_t(x)}.$$

Recognize Bayes $ p_t(x_0 | x) = \frac{p_t(x|x_0) p_t(x_0)}{p_t(x)}$, thus

$$\nabla_{x} \log p_t(x) = \int_{x_0 } -\frac{x-a_t x_0}{b_t^2} p_t(x_0|x) dx_0 = -\frac{1}{b_t^2}\left(x-a_t\mathbb{E}[x_0 | x]\right)$$

--- 

### Sampling

**Step 1: Implied Markov transition**

The marginals $x_t = a_t x_0 + b_t \varepsilon_t$ imply a Markov step from $s$ to $t$ ($0<s<t<1$). Writing $x_t$ in terms of $x_{s}$:

$$x_t = \frac{a_t}{a_{s}} x_{s} + \sqrt{b_t^2 - \frac{a_t^2}{a_{s}^2}b_{s}^2} \, \varepsilon_{s\to t}, \qquad \varepsilon_{s\to t} \sim \mathcal{N}(0, I).$$


**DDPM Update**

By Bayes
$$p(x_{s}| x_t, x_0) = \frac{p(x_{t}| x_s, x_0) p (x_s | x_0)}{p(x_{t}| x_0)}.$$

Because we specified that the relation between $x_s$ and $x_t$ is Markovian, tha above simplifies to
$$p(x_{s}| x_t, x_0) = \frac{p(x_{t}| x_s) p (x_s | x_0)}{p(x_{t}| x_0)}.$$

After some computations we arrive at DDPM update

$$x_s = \frac{a_t b_s^2}{a_s b_t^2}\,x_t + \frac{a_s\!\left(b_t^2 - \frac{a_t^2}{a_s^2}b_s^2\right)}{b_t^2}\,x_0 + \sqrt{\frac{b_s^2\!\left(b_t^2 - \frac{a_t^2}{a_s^2}b_s^2\right)}{b_t^2}}\; z, \qquad z\sim\mathcal{N}(0,I).$$

### Noising

In [ ]:
G_b_schedule = lambda t: t
# TODO G_a_schedule = ...

In [ ]:
N_noise_levels = 5
ts = torch.linspace(0, 1, N_noise_levels)
b_seq = G_b_schedule(ts)

base_samples = G_sampled_distribution_to_optimize.unlabeled_x
noise_dir = torch.normal(0, 1, (1, *base_samples.shape))
noise = b_seq.reshape(-1, 1, 1) * noise_dir

records = []
fig, axs = plt.subplots(1, len(noise), figsize=(len(noise) * 6, 5))
for i in range(len(noise)):
    model_name = "Drift({:.2f}), Noise({:.2f})".format(1.0, b_seq[i])
    noisy_data = base_samples + noise[i]
    metrics = G_sampled_distribution_to_optimize.compute_metrics(noisy_data)

    axs[i].set_title(model_name)
    viz_scatter(noisy_data, G_reward_fn, axs[i], cax_label="rewards")
    G_gt_distribution.viz_support(axs[i])

    metrics["model"] = model_name
    records.append(metrics)

pd.DataFrame.from_records(records, index="model")

### Denoising score matching

In [ ]:
G_noise_net = MLP(2 + 1, 64, 2)

In [ ]:
class NoisePredictionDataset:
    def __init__(
        self,
        data: torch.Tensor,
        # TODO: a_schedule: Callable[[torch.Tensor], torch.Tensor],
        b_schedule: Callable[[torch.Tensor], torch.Tensor],
        *,
        t_batch_size: int = 32,
        noise_dir_batch_size: int = 4,
    ):
        self.data = data
        self.b_schedule = b_schedule
        self.t_batch_size = t_batch_size
        self.noise_dir_batch_size = noise_dir_batch_size

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx: int):
        x = self.data[idx].reshape(1, 1, -1)
        t = torch.rand(size=(self.t_batch_size, 1, 1)).expand(
            (-1, self.noise_dir_batch_size, -1)
        )

        noise_level = self.b_schedule(t)
        noise_dir = torch.normal(
            0.0, 1.0, size=(1, self.noise_dir_batch_size, x.shape[-1])
        ).expand((self.t_batch_size, -1, -1))
        noise = noise_level * noise_dir

        noisy_x = x + noise

        return (
            torch.concatenate([noisy_x, t], axis=-1),
            noise,
        )

In [ ]:
from sklearn.model_selection import train_test_split

train_x, valid_x = train_test_split(
    G_sampled_distribution_to_optimize.unlabeled_x, test_size=0.2
)

train_data = NoisePredictionDataset(train_x, b_schedule=G_b_schedule)
valid_data = NoisePredictionDataset(valid_x, b_schedule=G_b_schedule)

fit(
    G_noise_net,
    loss_fn=lambda x, y: torch.nn.functional.mse_loss(x.flatten(), y.flatten()),
    optimizer=torch.optim.AdamW(
        G_noise_net.parameters(),
        lr=0.01,
    ),
    train_data=train_data,
    valid_data=valid_data,
    max_n_epochs=1000,
    patience=100,
)

### Denoised sample prediction

In [ ]:
def pred_x0_from_noise(
    noise_net: nn.Module,
    xt: torch.Tensor,
    t: torch.Tensor,
    a_schedule: Callable[[torch.Tensor], torch.Tensor],
    b_schedule: Callable[[torch.Tensor], torch.Tensor],
):
    xt = torch.as_tensor(xt, dtype=torch.float32)
    t = torch.as_tensor(t, dtype=torch.float32)
    t = torch.broadcast_to(t, (*xt.shape[:-1], 1))
    a_t = a_schedule(t)
    b_t = b_schedule(t)
    net_x = torch.concatenate([xt, t], axis=-1)
    noise_pred: torch.Tensor = noise_net(net_x)
    x0_pred = (xt - b_t * noise_pred) / a_t
    return x0_pred

### Denoising visualization

In [ ]:
%matplotlib widget
from matplotlib.widgets import Slider

x1_min = min(G_x1_min_display, -G_x1_max_display)
x1_max = max(G_x1_max_display, -G_x1_min_display)
x2_min = min(G_x2_min_display, -G_x2_max_display)
x2_max = max(G_x2_max_display, -G_x2_min_display)

grid_res = 20
x1_range = torch.linspace(x1_min, x1_max, grid_res)
x2_range = torch.linspace(x2_min, x2_max, grid_res)
X1, X2 = torch.meshgrid(x1_range, x2_range, indexing="xy")
x = torch.stack([X1.ravel(), X2.ravel()], axis=-1)

initial_t = 0.1

fig, ax = plt.subplots(figsize=(7, 7.5))
title_text = ax.set_title(
    rf"Denoising Vectors for noise scale $\sigma$ = {G_b_schedule(initial_t):.2f}"
)
G_gt_distribution.viz_support(ax)

quiver_plot = None

plt.subplots_adjust(bottom=0.2)

ax.set_xlim(x1_min, x1_max)
ax.set_ylim(x2_min, x2_max)
ax.set_xlabel("x1")
ax.set_ylabel("x2")
ax.grid(True, linestyle="--", alpha=0.5)

ax_slider = plt.axes([0.15, 0.07, 0.7, 0.03])
noise_slider = Slider(
    ax=ax_slider,
    label="Noise Level ($\sigma$)",
    valmin=0.0,
    valmax=1.0,
    valinit=initial_t,
    valfmt="%1.2f",
    color="crimson",
)


def update(t):
    global quiver_plot, title_text
    noise_level = G_b_schedule(t)
    G_noise_net.eval()

    with torch.no_grad():
        denoising_update = (
            pred_x0_from_noise(
                G_noise_net, x, t, lambda t: torch.full_like(t, 1.0), G_b_schedule
            )
            - x
        )

    if quiver_plot is None:
        quiver_plot = ax.quiver(
            x[:, 0],
            x[:, 1],
            denoising_update[:, 0],
            denoising_update[:, 1],
            angles="xy",
            scale_units="xy",
            color="crimson",
            alpha=0.3,
            zorder=2,
            scale=1.0,
        )
    else:
        quiver_plot.set_offsets(x)
        quiver_plot.set_UVC(denoising_update[:, 0], denoising_update[:, 1])

    title_text.set_text(
        rf"Denoising Vectors for noise scale $\sigma$ = {noise_level:.2f}"
    )
    fig.canvas.draw_idle()


update(initial_t)

noise_slider.on_changed(update)
plt.show()

In [ ]:
plt.close("all")
%matplotlib inline

### Iterative denoising

In [ ]:
def pred_x0_ddpm(
    noise_net: nn.Module,
    x1: torch.Tensor,
    a_schedule: Callable[[torch.Tensor], torch.Tensor],
    b_schedule: Callable[[torch.Tensor], torch.Tensor],
    N_steps: int,
):
    ts = torch.linspace(1.0, 0, N_steps + 1)
    drift = a_schedule(ts)
    noise_levels = b_schedule(ts)

    x_t = x1
    for i in range(0, N_steps):
        t = ts[i]
        a_t = drift[i]
        b_t = noise_levels[i]
        a_s = drift[i + 1]
        b_s = noise_levels[i + 1]

        x0_pred = pred_x0_from_noise(noise_net, x_t, t, a_schedule, b_schedule)

        bt2, bs2 = b_t**2, b_s**2
        sigma2 = bt2 - (a_t / a_s) ** 2 * bs2
        mean = (a_t * bs2) / (a_s * bt2) * x_t + (a_s * sigma2) / bt2 * x0_pred
        var = bs2 * sigma2 / bt2
        x_t = mean + var.sqrt() * torch.randn_like(x_t)

    return x_t

In [ ]:
x1_sample = G_sampled_distribution_to_optimize.unlabeled_x + G_b_schedule(
    1.0
) * torch.randn_like(G_sampled_distribution_to_optimize.unlabeled_x)

G_noise_net.eval()

metrics = G_sampled_distribution_to_optimize.compute_metrics(
    G_sampled_distribution_to_optimize.unlabeled_x
)
metrics["model"] = "initial_data"
records = [metrics]

N_steps_regimes = list(range(1, 6))
fig, axs = plt.subplots(1, len(N_steps_regimes), figsize=(len(N_steps_regimes) * 6, 5))
for i, n_steps in enumerate(N_steps_regimes):
    model_name = f"N_steps({n_steps})"
    with torch.no_grad():
        x0_pred = pred_x0_ddpm(
            G_noise_net,
            x1_sample,
            lambda t: torch.full_like(t, 1.0),
            G_b_schedule,
            n_steps,
        )
    metrics = G_sampled_distribution_to_optimize.compute_metrics(x0_pred)

    axs[i].set_title(model_name)
    viz_scatter(x0_pred, G_reward_fn, axs[i], cax_label="rewards")
    G_gt_distribution.viz_support(axs[i])

    metrics["model"] = model_name
    records.append(metrics)

pd.DataFrame.from_records(records, index="model")

## Classifier Guidance

To maximize our reward function during generation, we leverage Surrogate Guidance (analogous to Classifier Guidance in image diffusion). We want to sample from a modified distribution that tilts the original density toward high rewards:

$$p_{\text{guided}}(x) \propto p(x) \cdot R(x)^\gamma$$

Taking the log gradient (score) of this target distribution yields:$$\nabla_x \log p_{\text{guided}}(x) = \nabla_x \log p(x) + \gamma \nabla_x \log R(x).$$

Tweedie's formula gives a bijection between the score and the $x_0$ prediction:

$$\nabla_{x_t} \log p_t(x_t) = -\frac{1}{b_t^2}\bigl(x_t - a_t\,\mathbb{E}[x_0|x_t]\bigr) \quad\Longleftrightarrow\quad \mathbb{E}[x_0|x_t] = \frac{x_t + b_t^2\,\nabla_{x_t}\log p_t(x_t)}{a_t}$$

Applying the same inversion to the guided score:

$$\hat{x}_0^{\text{guided}} = \frac{x_t + b_t^2\,\nabla_{x_t}\log p_t^{\text{guided}}(x_t)}{a_t} = \frac{x_t + b_t^2\!\left(\nabla_{x_t}\log p_t(x_t) + \gamma\,\nabla_{x_t}\log R(x_t)\right)}{a_t}$$

$$\boxed{\hat{x}_0^{\text{guided}} = \underbrace{\mathbb{E}[x_0|x_t]}_{\text{denoiser output}} + \frac{b_t^2}{a_t}\,\gamma\,\nabla_{x_t}\log R(x_t)}$$

### Training reward surrogate

In [ ]:
G_reward_net = MLP(2, 32, 1)
from sklearn.model_selection import train_test_split

train_x, valid_x, train_y, valid_y = train_test_split(
    G_sampled_distribution_to_optimize.labeled_x,
    G_sampled_distribution_to_optimize.labeled_y,
    test_size=0.2,
)

train_data = torch.utils.data.TensorDataset(train_x.float(), train_y.float())
valid_data = torch.utils.data.TensorDataset(valid_x.float(), valid_y.float())

fit(
    G_reward_net,
    loss_fn=lambda x, y: torch.nn.functional.mse_loss(x.flatten(), y.flatten()),
    optimizer=torch.optim.SGD(
        G_reward_net.parameters(), lr=0.001
    ),  # TODO: maybe better optimizer would help?
    train_data=train_data,
    valid_data=valid_data,
    max_n_epochs=1000,
    patience=20,
)
fig, axs = plt.subplots(1, 2, figsize=(12, 5))
axs[0].set_title("Reward ground-truth")
axs[1].set_title("Reward prediction")

viz_img(G_reward_fn, axs[0])
with torch.no_grad():
    viz_img(G_reward_net, axs[1])

### Defining the classifier guidance

In [ ]:
def guidance_score_fn(
    reward_net: nn.Module,
    noise_net: nn.Module,
    xt: torch.Tensor,
    t: torch.Tensor,
    a_schedule: Callable[[torch.Tensor], torch.Tensor],
    b_schedule: Callable[[torch.Tensor], torch.Tensor],
):
    xt_ = xt.clone()
    xt_.requires_grad_(True)
    with torch.enable_grad():
        x0 = pred_x0_from_noise(noise_net, xt_, t, a_schedule, b_schedule)
        reward = reward_net(x0)
    grad = torch.autograd.grad(reward, xt_, grad_outputs=torch.ones_like(reward))[0]
    return grad


def pred_x0_ddpm_guided(
    noise_net: nn.Module,
    x1: torch.Tensor,
    a_schedule: Callable[[torch.Tensor], torch.Tensor],
    b_schedule: Callable[[torch.Tensor], torch.Tensor],
    reward_net: nn.Module,
    guidance_schedule: Callable[[torch.Tensor], torch.Tensor],
    N_steps: int,
):
    ts = torch.linspace(1.0, 0, N_steps + 1)
    drift = a_schedule(ts)
    noise_levels = b_schedule(ts)
    gammas = guidance_schedule(ts)

    xt = x1
    for i in range(0, N_steps):
        t = ts[i]
        a_t = drift[i]
        b_t = noise_levels[i]
        a_s = drift[i + 1]
        b_s = noise_levels[i + 1]
        gamma = gammas[i]

        x0_pred = pred_x0_from_noise(noise_net, xt, t, a_schedule, b_schedule)

        grad = guidance_score_fn(reward_net, noise_net, xt, t, a_schedule, b_schedule)
        x0_pred = x0_pred + (b_t**2 / a_t) * gamma * grad

        bt2, bs2 = b_t**2, b_s**2
        sigma2 = bt2 - (a_t / a_s) ** 2 * bs2
        mean = (a_t * bs2) / (a_s * bt2) * xt + (a_s * sigma2) / bt2 * x0_pred
        var = bs2 * sigma2 / bt2
        xt = mean + var.sqrt() * torch.randn_like(xt)

    return xt

### Visualization and improved reward metrics

In [ ]:
G_reward_net.eval()
for p in G_reward_net.parameters():
    p.requires_grad_(False)

In [ ]:
# TODO: Tune parameters
N_steps = 1
max_guidance = 0.1
# TODO: end

# TODO: Sample from pure noise
x1_sample = G_sampled_distribution_to_optimize.unlabeled_x + G_b_schedule(
    1.0
) * torch.randn_like(G_sampled_distribution_to_optimize.unlabeled_x)
# TODO: end

G_noise_net.eval()

metrics = G_sampled_distribution_to_optimize.compute_metrics(
    G_sampled_distribution_to_optimize.unlabeled_x
)
metrics["model"] = "initial_data"
records = [metrics]

N_scales = 5
fig, axs = plt.subplots(1, N_scales, figsize=(N_scales * 6, 5))
for i, guidance_scale in enumerate(torch.linspace(0, max_guidance, N_scales)):
    model_name = "Guidnce({:.2})".format(guidance_scale.item())
    with torch.no_grad():
        x0_pred = pred_x0_ddpm_guided(
            G_noise_net,
            x1_sample,
            lambda t: torch.full_like(t, 1.0),  # TODO
            G_b_schedule,
            G_reward_net,
            lambda t: guidance_scale * t,
            N_steps,
        )

    metrics = G_sampled_distribution_to_optimize.compute_metrics(x0_pred)

    axs[i].set_title(model_name)
    viz_scatter(
        x0_pred,
        lambda x: G_reward_fn(x),
        axs[i],
        cax_label="rewards",
    )
    G_gt_distribution.viz_support(axs[i])

    metrics["model"] = model_name
    records.append(metrics)

pd.DataFrame.from_records(records, index="model")

## Assignment 1: Transitioning from Noise-Perturbed Data to Pure Noise Sampling

### Current Limitation
Look closely at the forward process currently implemented in your notebook. The initial state $x_1$ at the beginning of the reverse loop is created by taking real data points and adding noise to them `G_sampled_distribution_to_optimize.unlabeled_x + G_b_schedule(1.0) * torch.randn_like(...)`. This means your generator cannot create completely novel data from scratch; it requires an existing starting point.

### Task
Introduce global variable `G_a_schedule` and modify the code to use the schedule instead of predefined constant. Select $a_t$ and $b_t$ schedules that reproduce DDPM.